In [1]:
%load_ext autoreload
%autoreload 2

# Component 01 binned with the DC4 511 keV imaging binning

Bins the cosmic-photon simulation (component 01) exactly like the DC4 511 keV analysis, using the workflow's `dataio_511.yaml` unchanged:

| axis | binning |
|---|---|
| Em | [509, 513] keV, 1 bin |
| Phi | 0–180°, 60 × 3° |
| PsiChi | HEALPix NSIDE 16, RING |
| Time | 2217 bins of 3599.44 s, 1835487300.036206 – 1843467254.887687 s |

PsiChi is binned twice from the same events:
- **Galactic**: as in the DC4 511 imaging analysis, the extended-source response and the workflow's binned total background;
- **local** (spacecraft frame): as in the detector response `Response511…h5` and the NF density.

Binning is done with this repository's cosipy (`BinnedData`, defaults as in the workflow: sparse, no overflow tracking). Only events in 508–514 keV are kept before binning; events outside the Em axis [509, 513] keV cannot enter any bin, so the result is the same as binning the full file.

## 1. Setup

### 1.1 Imports

In [2]:
import sys
from pathlib import Path

import numpy as np
import h5py
import astropy.units as u
from astropy.io import fits
from histpy import Histogram

from cosipy.data_io import BinnedData

sys.path.insert(0, str(Path.cwd().parent))   # project root, so `src` is importable
from src.plot_style import set_global_style
set_global_style()

### 1.2 Configuration

The DC4 binning yaml, the two responses and the binned total background are read in place (read-only) from the workflow repository. The outputs are new files; histpy refuses to overwrite an existing file.

In [3]:
# DC4 511 keV analysis files, used in place (read-only) from the workflow repository
WORKFLOW_DATA = Path("/Users/maura/Documents/COSI/data-challenge-4/workflow/data")
BINNING_YAML = WORKFLOW_DATA / "config/dataio_511.yaml"
RESPONSE_LOCAL = WORKFLOW_DATA / ("inputs/Response511.o4.e509_513.s20881894470591.m2555"
                                  ".filtered.nonsparse.binnedimaging.imagingresponse.h5")
RESPONSE_GALACTIC = WORKFLOW_DATA / "inputs/extended_source_response_511_merged.h5"
TOTAL_BKG_BINNED = WORKFLOW_DATA / "outputs/simulated_total_background_511_binned.hdf5"

# Component 01 (cosmic photons), uncompressed copy
SIM_PATH = Path("../data/background_components/"
                "CosmicPhotons_3months_unbinned_data_filtered_with_SAAcut.fits")

# Outputs: new files, histpy refuses to overwrite
FRAMES = ("galactic", "local")
OUT_PATHS = {frame: Path(f"../data/background_components/CosmicPhotons_511keV_DC4binning_{frame}.hdf5")
             for frame in FRAMES}

# Energy pre-selection only: wider than the Em axis [509, 513] keV, so the histogram edges decide
E_PRE_LO, E_PRE_HI = 508.0, 514.0
CHUNK = 1_000_000
COLUMNS = ("TimeTags", "Energies", "Phi", "Chi local", "Psi local", "Chi galactic", "Psi galactic")

## 2. Read component 01 and pre-select the energy window

The 10.9 GB file is read through a memory map in chunks of `CHUNK` rows (a few minutes). Expected: about 200k events in 508–514 keV (notebook 07 found 202,956 in 508–513 keV). Units: `Phi`, `Chi local`, `Psi local` in rad; `Chi galactic`, `Psi galactic` in deg.

In [4]:
with fits.open(SIM_PATH, memmap=True) as hdul:
    ev = hdul[1].data
    n_file = len(ev)
    sel = {k: [] for k in COLUMNS}
    for a in range(0, n_file, CHUNK):
        energy = ev["Energies"][a:a + CHUNK]
        keep = (energy >= E_PRE_LO) & (energy < E_PRE_HI)
        for k in COLUMNS:
            sel[k].append(np.array(ev[k][a:a + CHUNK][keep], dtype=np.float64))   # copy, native byte order
events = {k: np.concatenate(v) for k, v in sel.items()}

print(f"{n_file:,} events in file, {len(events['Energies']):,} in {E_PRE_LO:.0f}-{E_PRE_HI:.0f} keV")
for k, v in events.items():
    print(f"  {k:13s} [{v.min():.6f}, {v.max():.6f}]")

97,554,902 events in file, 242,146 in 508-514 keV
  TimeTags      [1835487353.202117, 1843466698.129811]
  Energies      [508.000000, 513.999800]
  Phi           [0.192847, 3.136417]
  Chi local     [0.000000, 6.282732]
  Psi local     [0.000000, 3.141593]
  Chi galactic  [0.000720, 359.999115]
  Psi galactic  [-89.849144, 89.938694]


### 2.1 Time coverage against the yaml

`n_in_range` is the number of events that should end up in the histograms (up to events lying exactly on a bin edge).

In [5]:
cfg = BinnedData(input_yaml=str(BINNING_YAML))
t, E = events["TimeTags"], events["Energies"]
print(f"yaml: tmin = {cfg.tmin}, tmax = {cfg.tmax}, time_bins = {cfg.time_bins} s")
print(f"component 01 (pre-selected): t in [{t.min():.6f}, {t.max():.6f}]")
print(f"pre-selected events outside [tmin, tmax]: {np.sum((t < cfg.tmin) | (t > cfg.tmax)):,}")

n_in_range = np.sum((E >= 509.0) & (E < 513.0) & (t >= cfg.tmin) & (t < cfg.tmax))
print(f"events in 509-513 keV and [tmin, tmax): {n_in_range:,}")

yaml: tmin = 1835487300.036206, tmax = 1843467254.887687, time_bins = 3600.0 s
component 01 (pre-selected): t in [1835487353.202117, 1843466698.129811]
pre-selected events outside [tmin, tmax]: 0
events in 509-513 keV and [tmin, tmax): 162,652


## 3. Bin in the Galactic and the local frame

`get_binned_data` uses `cosi_dataset` when no file is given, so the pre-selected events are binned directly. Both histograms must have the same total, equal to `n_in_range`.

In [6]:
binned = {}
for frame in FRAMES:
    bd = BinnedData(input_yaml=str(BINNING_YAML))
    bd.cosi_dataset = events                   # pre-selected events instead of re-reading the file
    bd.get_binned_data(psichi_binning=frame)   # defaults as in the workflow: sparse, no overflow tracking
    binned[frame] = bd.binned_data
    print(f"{frame:9s} axes {list(binned[frame].axes.labels)}, shape {binned[frame].contents.shape}, "
          f"total {float(binned[frame].contents.sum()):,.0f}")

fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking
fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


galactic  axes [np.str_('Time'), np.str_('Em'), np.str_('Phi'), np.str_('PsiChi')], shape (2217, 1, 60, 3072), total 162,652
local     axes [np.str_('Time'), np.str_('Em'), np.str_('Phi'), np.str_('PsiChi')], shape (2217, 1, 60, 3072), total 162,652


## 4. Check the binning against the responses and the workflow total background

Only the stored axes are read; the response matrices are not loaded. Every comparison must print `True`, and NSIDE, scheme and frame must match:
- local histogram vs detector response `Response511…h5` (`/DRM/AXES`, spacecraft frame);
- Galactic histogram vs extended-source response (`/hist/axes`, Galactic);
- Galactic histogram vs the workflow's binned total background (Time, Em, Phi, PsiChi).

In [7]:
def h5_axis(path, name):
    # Values and attributes of one stored histpy axis; the response matrix itself is not read.
    with h5py.File(path, "r") as f:
        return f[name][:], dict(f[name].attrs)


same = lambda a, b: np.allclose(a, b, rtol=0, atol=1e-6)

for frame, (path, group) in {"local": (RESPONSE_LOCAL, "/DRM/AXES"),
                             "galactic": (RESPONSE_GALACTIC, "/hist/axes")}.items():
    h = binned[frame]
    em, _ = h5_axis(path, f"{group}/2")
    phi, _ = h5_axis(path, f"{group}/3")
    _, psichi = h5_axis(path, f"{group}/4")
    ax = h.axes["PsiChi"]
    print(f"--- {frame} histogram vs {path.name}")
    print("  Em edges equal :", same(h.axes["Em"].edges.to_value(u.keV), em))
    print("  Phi edges equal:", same(h.axes["Phi"].edges.to_value(u.deg), phi))
    print(f"  PsiChi binned  : nside={ax.nside}, scheme={ax.scheme}, coordsys={ax.coordsys}")
    print(f"  PsiChi response: nside={psichi['nside']}, scheme={psichi['scheme']}, coordsys={psichi['coordsys']}")

total_bkg = Histogram.open(TOTAL_BKG_BINNED)
print(f"--- galactic histogram vs {TOTAL_BKG_BINNED.name}")
for label in ("Time", "Em", "Phi"):
    print(f"  {label} edges equal:", same(binned["galactic"].axes[label].edges.value,
                                          total_bkg.axes[label].edges.value))
ax = total_bkg.axes["PsiChi"]
print(f"  PsiChi total bkg: nside={ax.nside}, scheme={ax.scheme}, coordsys={ax.coordsys}")

n01, ntot = float(binned["galactic"].contents.sum()), float(total_bkg.contents.sum())
print(f"component 01 / total background in 509-513 keV: {n01:,.0f} / {ntot:,.0f} = {n01 / ntot:.4f}")

--- local histogram vs Response511.o4.e509_513.s20881894470591.m2555.filtered.nonsparse.binnedimaging.imagingresponse.h5
  Em edges equal : True
  Phi edges equal: True
  PsiChi binned  : nside=16, scheme=RING, coordsys=<SpacecraftFrame Frame (attitude=None, obstime=None, location=None)>
  PsiChi response: nside=16, scheme=RING, coordsys=spacecraftframe
--- galactic histogram vs extended_source_response_511_merged.h5
  Em edges equal : True
  Phi edges equal: True
  PsiChi binned  : nside=16, scheme=RING, coordsys=<Galactic Frame>
  PsiChi response: nside=16, scheme=RING, coordsys=galactic
--- galactic histogram vs simulated_total_background_511_binned.hdf5
  Time edges equal: True
  Em edges equal: True
  Phi edges equal: True
  PsiChi total bkg: nside=16, scheme=RING, coordsys=<Galactic Frame>
component 01 / total background in 509-513 keV: 162,652 / 1,374,985 = 0.1183


## 5. Write the binned histograms

Two new files in `data/background_components/` (gitignored). histpy's default `overwrite=False` raises an error if a file already exists.

In [ ]:
def write_new(hist, path):
    # Write a histogram only if the file does not exist yet (never overwrite).
    if path.exists():
        print("exists, not overwritten:", path.resolve())
    else:
        hist.write(path)
        print("written:", path.resolve())


for frame in FRAMES:
    write_new(binned[frame], OUT_PATHS[frame])

## 6. NF samples (caffeinated) binned the same way

`part-01_full_caffeinated.hdf5` holds 97,554,902 events sampled from the same part-01 NF model, with only `Energies` [keV], `Phi`, `Chi local`, `Psi local` [rad]: no times and no Galactic coordinates. They are binned in the **local frame only, without a Time axis**:
- the axes are copied from the local simulation histogram (Section 3) summed over time, so Em, Phi and PsiChi are identical;
- the histogram is filled exactly as `BinnedData` does for local binning (lon = `Chi local`, lat = π/2 − `Psi local`, Phi in deg); `BinnedData` itself needs `TimeTags`, so it is not used here;
- the same 508–514 keV pre-selection and `CHUNK` reading as in Section 2 (about a minute).

In [ ]:
from astropy.coordinates import SkyCoord
from scoords import SpacecraftFrame

NF_SAMPLES_PATH = Path("../data/NF_background/part-01_full_caffeinated.hdf5")
NF_OUT_PATH = Path("../data/NF_background/part-01_full_caffeinated_511keV_DC4binning_local.hdf5")
NF_COLUMNS = ("Energies", "Phi", "Chi local", "Psi local")

# Read and pre-select the energy window
with h5py.File(NF_SAMPLES_PATH, "r") as f:
    print("model:", f.attrs["model_file"], "|", f.attrs["description"])
    n_nf = f["Energies"].shape[0]
    sel = {k: [] for k in NF_COLUMNS}
    for a in range(0, n_nf, CHUNK):
        energy = f["Energies"][a:a + CHUNK]
        keep = (energy >= E_PRE_LO) & (energy < E_PRE_HI)
        for k in NF_COLUMNS:
            sel[k].append(np.asarray(f[k][a:a + CHUNK][keep], dtype=np.float64))
nf_events = {k: np.concatenate(v) for k, v in sel.items()}
print(f"{n_nf:,} NF samples, {len(nf_events['Energies']):,} in {E_PRE_LO:.0f}-{E_PRE_HI:.0f} keV")

# Same Em, Phi, PsiChi (local) axes as the simulation; filled as BinnedData does for local binning
sim_local = binned["local"].project("Em", "Phi", "PsiChi")
nf_binned = Histogram(sim_local.axes)
coords = SkyCoord(lon=nf_events["Chi local"], lat=np.pi / 2 - nf_events["Psi local"],
                  unit=u.rad, frame=SpacecraftFrame())
nf_binned.fill(u.Quantity(nf_events["Energies"], u.keV),
               u.Quantity(np.rad2deg(nf_events["Phi"]), u.deg),
               coords)

E_nf = nf_events["Energies"]
print(f"NF samples in 509-513 keV: {np.sum((E_nf >= 509.0) & (E_nf < 513.0)):,}")
print(f"NF binned:  axes {list(nf_binned.axes.labels)}, shape {nf_binned.contents.shape}, "
      f"total {float(nf_binned.contents.sum()):,.0f}")
print(f"simulation (local, summed over time): total {float(sim_local.contents.sum()):,.0f}")

write_new(nf_binned, NF_OUT_PATH)

## 7. NF samples in the Galactic frame

The samples have no times, so they are rotated as a histogram, with the same method as the NF template in `src/nf_binned_template.py` (`galactic_conversion_matrix`, the method cosipy uses for its Galactic responses):
- `W[g, l]` = live time [s] in [tmin, tmax] during which the center of Galactic pixel g lies in local pixel l (attitudes and live time from the 15 s orientation file);
- Galactic counts = (local counts / total live time) @ Wᵀ. This assumes the samples, like the simulation, correspond to the full window [tmin, tmax] (97,554,902 samples = events in the simulation file).

**Check of the rotation.** The simulation is rotated the same way from its local histogram and compared with its Galactic histogram binned from the per-event Galactic coordinates (Section 3). If the rotation is right, the two sky maps (summed over Em and Phi) agree: the pulls (direct − rotated)/√rotated scatter around 0 with a width at most ~1 (the two maps come from the same events). A wrong rotation gives clearly different maps and pulls much wider than 1.

In [ ]:
from astropy.time import Time
from cosipy.spacecraftfile import SpacecraftHistory
from src.nf_binned_template import galactic_conversion_matrix

ORI_PATH = Path("../data/DC4_orientation.ori")   # same file as DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori (same MD5)
NF_OUT_PATH_GAL = Path("../data/NF_background/part-01_full_caffeinated_511keV_DC4binning_galactic.hdf5")

# Attitude and live time in the binning window [tmin, tmax] of the yaml
history = SpacecraftHistory.open(ORI_PATH).select_interval(Time(cfg.tmin, format="unix"),
                                                           Time(cfg.tmax, format="unix"))
livetime = float(history.livetime.to_value(u.s).sum())

psichi = nf_binned.axes["PsiChi"]
W = galactic_conversion_matrix(history, psichi.nside, nest=(psichi.scheme == "NESTED"))   # [s], (3072, 3072)
print(f"live time in [tmin, tmax]: {livetime:,.1f} s | every row of W sums to it: "
      f"{np.allclose(W.sum(axis=1), livetime)}")

gal_axes = binned["galactic"].project("Em", "Phi", "PsiChi").axes   # Em, Phi, PsiChi (Galactic) of the simulation


def to_galactic(hist_local):
    # Local-frame counts over [tmin, tmax] -> Galactic-frame counts: (counts / live time) @ W^T.
    counts_local = np.asarray(hist_local.to_dense().contents)        # (1, 60, 3072)
    return Histogram(gal_axes, contents=(counts_local / livetime) @ W.T)


nf_galactic = to_galactic(nf_binned)
print(f"NF samples: local total {float(nf_binned.contents.sum()):,.0f} -> "
      f"Galactic total {float(nf_galactic.contents.sum()):,.0f}, shape {nf_galactic.contents.shape}")

write_new(nf_galactic, NF_OUT_PATH_GAL)

### 7.1 Check of the rotation with the simulation

Sky maps summed over Em and Phi: simulation binned from its per-event Galactic coordinates (direct) vs simulation rotated from its local histogram with `W` (rotated), and the pulls.

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt

sim_direct = np.asarray(binned["galactic"].project("PsiChi").to_dense().contents)   # per-event Galactic coordinates
sim_rotated = np.asarray(to_galactic(sim_local).project("PsiChi").contents)         # local histogram rotated with W
pull = (sim_direct - sim_rotated) / np.sqrt(np.where(sim_rotated > 0, sim_rotated, np.nan))

print(f"totals: direct {sim_direct.sum():,.0f} | rotated {sim_rotated.sum():,.1f}")
print(f"pulls: mean {np.nanmean(pull):+.3f}, std {np.nanstd(pull):.3f}, max |pull| {np.nanmax(np.abs(pull)):.2f}")

fig = plt.figure(figsize=(18, 4.5))
hp.mollview(sim_direct, fig=fig.number, sub=(1, 3, 1), unit="counts",
            title="simulation, direct (per-event Galactic)")
hp.mollview(sim_rotated, fig=fig.number, sub=(1, 3, 2), unit="counts",
            title="simulation, rotated with W")
hp.mollview(pull, fig=fig.number, sub=(1, 3, 3), min=-4, max=4, cmap="RdBu_r",
            title="pull (direct - rotated) / sqrt(rotated)")
plt.show()